In [3]:
from rdflib import Graph, Namespace
from rdflib.namespace import SKOS
import pandas as pd
from rdflib import URIRef


In [ ]:
# Correct RDF URL for EU Languages
def getLanguage(): 
    LANGUAGE_RDF_URL = "https://publications.europa.eu/resource/authority/language"

    # Load RDF
    print("Loading language vocabulary...")
    g = Graph()
    g.parse(LANGUAGE_RDF_URL,format="xml")

    # Extract English labels and URIs
    entries = []
    for s, p, o in g.triples((None, SKOS.prefLabel, None)):
        print(s,p,o)
        if o.language == "english":
            entries.append({"label": str(o), "uri": str(s)})

    # Create DataFrame
    df = pd.DataFrame(entries)

    # Lookup function
    def lookup_language(term):
        matches = df[df["label"].str.contains(term, case=False)]
        if not matches.empty:
            print(matches.to_string(index=False))
        else:
            print(f"No match for '{term}'")

    # Try searching
    print("\n🔎 Search example for 'English':")
    lookup_language("eng")

# Optional: Save to CSV
#df.to_csv("language_vocab.csv", index=False)


Loading language vocabulary...
http://publications.europa.eu/resource/authority/language http://www.w3.org/2004/02/skos/core#prefLabel Language

🔎 Search example for 'french':


KeyError: 'label'

In [3]:
g

<Graph identifier=Neabc5b67f3fe469da1ae966aa3192abe (<class 'rdflib.graph.Graph'>)>

In [4]:
g.triples((None, SKOS.prefLabel, None))

<generator object Graph.triples at 0x77501245dfc0>

In [7]:
for s, p, o in g.triples((None, SKOS.prefLabel, None)):
    print(s,p,o)

http://publications.europa.eu/resource/authority/language http://www.w3.org/2004/02/skos/core#prefLabel Language


In [9]:
# Let's investigate the language attributes of the labels
print("Investigating language attributes:")
count = 0
for s, p, o in g.triples((None, SKOS.prefLabel, None)):
    print(f"Subject: {s}")
    print(f"Predicate: {p}")
    print(f"Object: {o}")
    print(f"Object language: {o.language}")
    print(f"Object type: {type(o)}")
    print("---")
    count += 1
    if count >= 10:  # Limit output to first 10
        print("(showing first 10 results...)")
        break

Investigating language attributes:
Subject: http://publications.europa.eu/resource/authority/language
Predicate: http://www.w3.org/2004/02/skos/core#prefLabel
Object: Language
Object language: en
Object type: <class 'rdflib.term.Literal'>
---


In [10]:
# Let's check the total number of triples and different types
print(f"Total triples in graph: {len(g)}")
print()

# Count prefLabel triples
prefLabel_count = 0
for s, p, o in g.triples((None, SKOS.prefLabel, None)):
    prefLabel_count += 1

print(f"Total prefLabel triples: {prefLabel_count}")
print()

# Let's see what other predicates exist
predicates = set()
for s, p, o in g.triples((None, None, None)):
    predicates.add(p)

print(f"Unique predicates ({len(predicates)}):")
for pred in sorted(predicates):
    print(f"  {pred}")

Total triples in graph: 24492

Total prefLabel triples: 1

Unique predicates (18):
  http://publications.europa.eu/ontology/authority/prefLabel
  http://publications.europa.eu/ontology/authority/table.id
  http://publications.europa.eu/ontology/authority/table.version.number
  http://publications.europa.eu/ontology/euvoc#startDate
  http://purl.org/dc/terms/created
  http://purl.org/dc/terms/identifier
  http://purl.org/dc/terms/title
  http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  http://www.w3.org/2000/01/rdf-schema#comment
  http://www.w3.org/2000/01/rdf-schema#label
  http://www.w3.org/2002/07/owl#imports
  http://www.w3.org/2002/07/owl#versionIRI
  http://www.w3.org/2002/07/owl#versionInfo
  http://www.w3.org/2004/02/skos/core#hasTopConcept
  http://www.w3.org/2004/02/skos/core#inScheme
  http://www.w3.org/2004/02/skos/core#prefLabel
  http://www.w3.org/2004/02/skos/core#topConceptOf
  http://www.w3.org/2008/05/skos-xl#prefLabel


In [11]:
from rdflib import URIRef

# Let's look for subjects that might be language concepts
print("Looking for language concept subjects:")
language_subjects = set()

# Get all subjects
for s, p, o in g.triples((None, None, None)):
    if "language" in str(s).lower():
        language_subjects.add(s)

print(f"Found {len(language_subjects)} subjects containing 'language':")
for subj in sorted(language_subjects):
    print(f"  {subj}")

print("\n" + "="*50)

# Let's examine what properties the main language authority has
main_lang_uri = "http://publications.europa.eu/resource/authority/language"
print(f"Properties of {main_lang_uri}:")
for s, p, o in g.triples((URIRef(main_lang_uri), None, None)):
    print(f"  {p} -> {o}")

Looking for language concept subjects:
Found 8189 subjects containing 'language':
  http://publications.europa.eu/resource/authority/language
  http://publications.europa.eu/resource/authority/language/0D0
  http://publications.europa.eu/resource/authority/language/0E0
  http://publications.europa.eu/resource/authority/language/AAA
  http://publications.europa.eu/resource/authority/language/AAB
  http://publications.europa.eu/resource/authority/language/AAC
  http://publications.europa.eu/resource/authority/language/AAD
  http://publications.europa.eu/resource/authority/language/AAE
  http://publications.europa.eu/resource/authority/language/AAF
  http://publications.europa.eu/resource/authority/language/AAG
  http://publications.europa.eu/resource/authority/language/AAH
  http://publications.europa.eu/resource/authority/language/AAI
  http://publications.europa.eu/resource/authority/language/AAK
  http://publications.europa.eu/resource/authority/language/AAL
  http://publications.euro

In [12]:
# Let's look at the structure differently - check if there are individual language concept URIs
print("Looking for individual language concept URIs (not the main authority):")

# Look for subjects that have language-related patterns but are not the main authority
individual_languages = set()
for s, p, o in g.triples((None, None, None)):
    s_str = str(s)
    if ("language" in s_str.lower() and 
        s_str != "http://publications.europa.eu/resource/authority/language" and
        "/language/" in s_str):
        individual_languages.add(s)

print(f"Found {len(individual_languages)} individual language concept URIs")

# Show a few examples
count = 0
for lang_uri in sorted(individual_languages):
    if count < 5:
        print(f"\nExamining: {lang_uri}")
        # Get all properties for this language
        props = []
        for s, p, o in g.triples((lang_uri, None, None)):
            props.append((p, o))
        
        print(f"  Has {len(props)} properties:")
        for p, o in props[:3]:  # Show first 3 properties
            print(f"    {p} -> {o}")
        if len(props) > 3:
            print("    ...")
        count += 1
    else:
        break

Looking for individual language concept URIs (not the main authority):
Found 8188 individual language concept URIs

Examining: http://publications.europa.eu/resource/authority/language/0D0
  Has 2 properties:
    http://www.w3.org/2004/02/skos/core#inScheme -> http://publications.europa.eu/resource/authority/language
    http://www.w3.org/2004/02/skos/core#topConceptOf -> http://publications.europa.eu/resource/authority/language

Examining: http://publications.europa.eu/resource/authority/language/0E0
  Has 2 properties:
    http://www.w3.org/2004/02/skos/core#inScheme -> http://publications.europa.eu/resource/authority/language
    http://www.w3.org/2004/02/skos/core#topConceptOf -> http://publications.europa.eu/resource/authority/language

Examining: http://publications.europa.eu/resource/authority/language/AAA
  Has 2 properties:
    http://www.w3.org/2004/02/skos/core#inScheme -> http://publications.europa.eu/resource/authority/language
    http://www.w3.org/2004/02/skos/core#topCo

In [2]:
# Let's check what types of label properties exist for individual language concepts
from rdflib.namespace import RDFS

print("Checking for different label properties on language concepts:")

# Define the different label predicates we found
AUTH_PREFLABEL = URIRef("http://publications.europa.eu/ontology/authority/prefLabel")
SKOS_XL_PREFLABEL = URIRef("http://www.w3.org/2008/05/skos-xl#prefLabel")

# Test with a few language concepts
test_langs = [
    "http://publications.europa.eu/resource/authority/language/ENG",  # English
    "http://publications.europa.eu/resource/authority/language/FRA",  # French
    "http://publications.europa.eu/resource/authority/language/DEU",  # German
]

for lang_uri_str in test_langs:
    lang_uri = URIRef(lang_uri_str)
    print(f"\nChecking {lang_uri_str}:")
    
    # Check SKOS prefLabel
    for s, p, o in g.triples((lang_uri, SKOS.prefLabel, None)):
        print(f"  SKOS prefLabel: {o} (lang: {o.language})")
    
    # Check authority prefLabel
    for s, p, o in g.triples((lang_uri, AUTH_PREFLABEL, None)):
        print(f"  Authority prefLabel: {o} (lang: {getattr(o, 'language', 'no lang')})")
        
    # Check RDFS label
    for s, p, o in g.triples((lang_uri, RDFS.label, None)):
        print(f"  RDFS label: {o} (lang: {getattr(o, 'language', 'no lang')})")
    
    # Check SKOS-XL prefLabel
    for s, p, o in g.triples((lang_uri, SKOS_XL_PREFLABEL, None)):
        print(f"  SKOS-XL prefLabel: {o} (lang: {getattr(o, 'language', 'no lang')})")
        
    # Check all properties for this language
    all_props = list(g.triples((lang_uri, None, None)))
    print(f"  Total properties: {len(all_props)}")
    if len(all_props) == 0:
        print("  (This language concept doesn't exist in the dataset)")

Checking for different label properties on language concepts:


NameError: name 'URIRef' is not defined

In [6]:
for s, p, o in g.triples():
    print(s, p, o)

TypeError: Graph.triples() missing 1 required positional argument: 'triple'

In [13]:
# Comprehensive analysis to extract proper language data for DCAT 3
from rdflib.namespace import RDFS, DCTERMS
from rdflib import URIRef

print("=== DCAT 3 Language Data Analysis ===")

# Check what types of language concepts exist with actual labels
print("\n1. Looking for individual language concepts with labels...")

# Define different label predicates
AUTH_PREFLABEL = URIRef("http://publications.europa.eu/ontology/authority/prefLabel")

# Sample some known language codes
sample_language_codes = ["ENG", "FRA", "DEU", "SPA", "ITA", "POR", "NLD", "SWE", "DAN", "FIN"]

languages_found = []
for code in sample_language_codes:
    lang_uri = URIRef(f"http://publications.europa.eu/resource/authority/language/{code}")
    
    # Check if this URI exists and has labels
    labels = {}
    
    # Check authority prefLabel
    for s, p, o in g.triples((lang_uri, AUTH_PREFLABEL, None)):
        if o.language not in labels:
            labels[o.language] = []
        labels[o.language].append(("auth:prefLabel", str(o)))
    
    # Check SKOS prefLabel  
    for s, p, o in g.triples((lang_uri, SKOS.prefLabel, None)):
        if o.language not in labels:
            labels[o.language] = []
        labels[o.language].append(("skos:prefLabel", str(o)))
        
    # Check RDFS label
    for s, p, o in g.triples((lang_uri, RDFS.label, None)):
        lang_code = getattr(o, 'language', 'no-lang')
        if lang_code not in labels:
            labels[lang_code] = []
        labels[lang_code].append(("rdfs:label", str(o)))
    
    if labels:
        languages_found.append({
            "code": code,
            "uri": str(lang_uri),
            "labels": labels
        })

print(f"Found {len(languages_found)} languages with labels:")
for lang in languages_found[:5]:  # Show first 5
    print(f"\n  {lang['code']} ({lang['uri']}):")
    for lang_code, label_list in lang['labels'].items():
        for label_type, label_value in label_list:
            print(f"    {lang_code}: {label_type} = '{label_value}'")

=== DCAT 3 Language Data Analysis ===

1. Looking for individual language concepts with labels...
Found 0 languages with labels:


In [14]:
# Let's check the hasTopConcept relationships and actual language data structure
print("\n2. Analyzing the language authority structure...")

from rdflib.namespace import SKOS

main_authority = URIRef("http://publications.europa.eu/resource/authority/language")

# Check top concepts
print("Top concepts in the language authority:")
top_concepts = []
for s, p, o in g.triples((main_authority, SKOS.hasTopConcept, None)):
    top_concepts.append(o)

print(f"Found {len(top_concepts)} top concepts")

# Let's examine a few top concepts to see what data they contain
print("\nExamining top concepts structure:")
for i, concept in enumerate(top_concepts[:5]):
    print(f"\nTop concept {i+1}: {concept}")
    
    # Get all properties for this concept
    properties = list(g.triples((concept, None, None)))
    print(f"  Properties ({len(properties)}):")
    
    for s, p, o in properties:
        print(f"    {p} -> {o}")
        if hasattr(o, 'language') and o.language:
            print(f"      (language: {o.language})")

# Also check if there are any concepts that are topConceptOf the authority
print(f"\n3. Checking concepts that are topConceptOf the language authority...")
top_concept_of_count = 0
sample_concepts = []

for s, p, o in g.triples((None, SKOS.topConceptOf, main_authority)):
    top_concept_of_count += 1
    if len(sample_concepts) < 10:
        sample_concepts.append(s)

print(f"Found {top_concept_of_count} concepts that are topConceptOf language authority")

print(f"\nSample concepts:")
for concept in sample_concepts:
    print(f"  {concept}")
    
    # Check if any of these have labels
    has_labels = False
    for s, p, o in g.triples((concept, None, None)):
        if "label" in str(p).lower() or "preflabel" in str(p).lower():
            print(f"    {p} -> {o}")
            if hasattr(o, 'language'):
                print(f"      (language: {o.language})")
            has_labels = True
    if not has_labels:
        print("    (no labels found)")
    print()


2. Analyzing the language authority structure...
Top concepts in the language authority:
Found 8144 top concepts

Examining top concepts structure:

Top concept 1: http://publications.europa.eu/resource/authority/language/PNH
  Properties (2):
    http://www.w3.org/2004/02/skos/core#inScheme -> http://publications.europa.eu/resource/authority/language
    http://www.w3.org/2004/02/skos/core#topConceptOf -> http://publications.europa.eu/resource/authority/language

Top concept 2: http://publications.europa.eu/resource/authority/language/PNC
  Properties (2):
    http://www.w3.org/2004/02/skos/core#inScheme -> http://publications.europa.eu/resource/authority/language
    http://www.w3.org/2004/02/skos/core#topConceptOf -> http://publications.europa.eu/resource/authority/language

Top concept 3: http://publications.europa.eu/resource/authority/language/PNE
  Properties (2):
    http://www.w3.org/2004/02/skos/core#inScheme -> http://publications.europa.eu/resource/authority/language
    h

In [15]:
# CORRECTED APPROACH: Extract proper language data for DCAT 3
print("=== CORRECTED LANGUAGE DATA EXTRACTION FOR DCAT 3 ===")

# The issue is that the EU Publications Office RDF endpoint might not contain
# the actual language labels, just the authority metadata.
# For DCAT 3, we need the language codes and their proper labels.

# Let's create a proper language extraction function using the authority prefLabel
from rdflib.namespace import RDFS

AUTH_PREFLABEL = URIRef("http://publications.europa.eu/ontology/authority/prefLabel")

print("1. Extracting language concepts that have authority labels...")

languages_with_labels = []
processed_uris = set()

# Check all triples with authority prefLabel
for s, p, o in g.triples((None, AUTH_PREFLABEL, None)):
    if str(s) not in processed_uris and "/language/" in str(s):
        processed_uris.add(str(s))
        
        # Extract the language code from the URI
        language_code = str(s).split("/language/")[-1]
        
        # Get all labels for this language in different languages
        labels = {}
        for s2, p2, o2 in g.triples((s, AUTH_PREFLABEL, None)):
            lang = getattr(o2, 'language', 'no-lang')
            if lang not in labels:
                labels[lang] = []
            labels[lang].append(str(o2))

        if labels:
            languages_with_labels.append({
                "code": language_code,
                "uri": str(s),
                "labels": labels
            })

print(f"Found {len(languages_with_labels)} languages with authority labels")

# Display first 10 for inspection
for i, lang_data in enumerate(languages_with_labels[:10]):
    print(f"\n{i+1}. {lang_data['code']} ({lang_data['uri']}):")
    for lang_code, label_list in lang_data['labels'].items():
        for label in label_list:
            print(f"   {lang_code}: '{label}'")

print(f"\n(Showing first 10 of {len(languages_with_labels)} total languages)")

# Create the proper DataFrame for DCAT 3 usage
if languages_with_labels:
    print("\n2. Creating DCAT 3 compatible language DataFrame...")
    
    # Create entries for English labels (most commonly used in DCAT)
    dcat_entries = []
    for lang_data in languages_with_labels:
        # Prefer English labels, fall back to others if English not available
        english_label = None
        fallback_label = None
        
        if 'en' in lang_data['labels']:
            english_label = lang_data['labels']['en'][0]
        elif 'no-lang' in lang_data['labels']:
            fallback_label = lang_data['labels']['no-lang'][0]
        elif lang_data['labels']:  # Use any available label
            first_lang = list(lang_data['labels'].keys())[0]
            fallback_label = lang_data['labels'][first_lang][0]
            
        label_to_use = english_label or fallback_label
        
        if label_to_use:
            dcat_entries.append({
                "code": lang_data['code'],
                "label": label_to_use,
                "uri": lang_data['uri']
            })
    
    # Create DataFrame
    df_languages = pd.DataFrame(dcat_entries)
    print(f"Created DataFrame with {len(df_languages)} language entries")
    
    # Show sample
    print("\nSample entries:")
    print(df_languages.head(10).to_string(index=False))
    
    # Save to CSV for DCAT 3 usage
    df_languages.to_csv("dcat3_languages.csv", index=False)
    print(f"\n✅ Saved {len(df_languages)} language entries to 'dcat3_languages.csv'")
    
    # Create lookup function for DCAT 3
    def lookup_dcat_language(search_term):
        """Look up language by code or label for DCAT 3 usage"""
        matches = df_languages[
            (df_languages["code"].str.contains(search_term, case=False)) |
            (df_languages["label"].str.contains(search_term, case=False))
        ]
        return matches
    
    # Test the lookup
    print(f"\n3. Testing language lookup:")
    print("Search for 'ENG':")
    print(lookup_dcat_language("ENG").to_string(index=False))
    
    print("\nSearch for 'French':")
    print(lookup_dcat_language("French").to_string(index=False))
    
else:
    print("No languages with authority labels found in this dataset.")

=== CORRECTED LANGUAGE DATA EXTRACTION FOR DCAT 3 ===
1. Extracting language concepts that have authority labels...
Found 0 languages with authority labels

(Showing first 10 of 0 total languages)
No languages with authority labels found in this dataset.


In [16]:
# Alternative approach: Try different EU language authority endpoints
print("=== ALTERNATIVE APPROACH: Different Language Authority Endpoints ===")

# The current endpoint might just be metadata. Let's try the full vocabulary endpoint
alternative_urls = [
    "https://publications.europa.eu/resource/authority/language.rdf",
    "https://publications.europa.eu/resource/authority/language.xml",
    "https://op.europa.eu/en/web/eu-vocabularies/authority-tables",  # This is HTML, but let's see
]

print("Trying alternative endpoints...")

for url in alternative_urls:
    print(f"\nTrying: {url}")
    try:
        g_alt = Graph()
        g_alt.parse(url, format="xml")
        print(f"✅ Successfully loaded {len(g_alt)} triples")
        
        # Quick check for language concepts with labels
        lang_count = 0
        for s, p, o in g_alt.triples((None, SKOS.prefLabel, None)):
            if "/language/" in str(s):
                lang_count += 1
                if lang_count <= 5:  # Show first 5
                    print(f"  Found: {s} -> {o} (lang: {getattr(o, 'language', 'none')})")
        
        if lang_count > 0:
            print(f"✅ Found {lang_count} language concepts with SKOS prefLabels!")
            # Use this graph instead
            g = g_alt
            break
        else:
            print(f"❌ No language concepts with prefLabels found")
            
    except Exception as e:
        print(f"❌ Failed to load: {e}")

# If no alternative worked, let's create a basic DCAT 3 language mapping manually
print(f"\n=== MANUAL DCAT 3 LANGUAGE MAPPING ===")
print("Since the RDF endpoints may not provide complete vocabulary,")
print("creating a basic ISO 639-1/639-2 language mapping for DCAT 3:")

# Basic language mapping based on common DCAT 3 usage
basic_languages = [
    {"code": "en", "iso639_2": "ENG", "label": "English", "uri": "http://publications.europa.eu/resource/authority/language/ENG"},
    {"code": "fr", "iso639_2": "FRA", "label": "French", "uri": "http://publications.europa.eu/resource/authority/language/FRA"},
    {"code": "de", "iso639_2": "DEU", "label": "German", "uri": "http://publications.europa.eu/resource/authority/language/DEU"},
    {"code": "es", "iso639_2": "SPA", "label": "Spanish", "uri": "http://publications.europa.eu/resource/authority/language/SPA"},
    {"code": "it", "iso639_2": "ITA", "label": "Italian", "uri": "http://publications.europa.eu/resource/authority/language/ITA"},
    {"code": "pt", "iso639_2": "POR", "label": "Portuguese", "uri": "http://publications.europa.eu/resource/authority/language/POR"},
    {"code": "nl", "iso639_2": "NLD", "label": "Dutch", "uri": "http://publications.europa.eu/resource/authority/language/NLD"},
    {"code": "sv", "iso639_2": "SWE", "label": "Swedish", "uri": "http://publications.europa.eu/resource/authority/language/SWE"},
    {"code": "da", "iso639_2": "DAN", "label": "Danish", "uri": "http://publications.europa.eu/resource/authority/language/DAN"},
    {"code": "fi", "iso639_2": "FIN", "label": "Finnish", "uri": "http://publications.europa.eu/resource/authority/language/FIN"},
    {"code": "el", "iso639_2": "ELL", "label": "Greek", "uri": "http://publications.europa.eu/resource/authority/language/ELL"},
    {"code": "pl", "iso639_2": "POL", "label": "Polish", "uri": "http://publications.europa.eu/resource/authority/language/POL"},
    {"code": "cs", "iso639_2": "CES", "label": "Czech", "uri": "http://publications.europa.eu/resource/authority/language/CES"},
    {"code": "sk", "iso639_2": "SLK", "label": "Slovak", "uri": "http://publications.europa.eu/resource/authority/language/SLK"},
    {"code": "hu", "iso639_2": "HUN", "label": "Hungarian", "uri": "http://publications.europa.eu/resource/authority/language/HUN"},
    {"code": "ro", "iso639_2": "RON", "label": "Romanian", "uri": "http://publications.europa.eu/resource/authority/language/RON"},
    {"code": "bg", "iso639_2": "BUL", "label": "Bulgarian", "uri": "http://publications.europa.eu/resource/authority/language/BUL"},
    {"code": "hr", "iso639_2": "HRV", "label": "Croatian", "uri": "http://publications.europa.eu/resource/authority/language/HRV"},
    {"code": "sl", "iso639_2": "SLV", "label": "Slovenian", "uri": "http://publications.europa.eu/resource/authority/language/SLV"},
    {"code": "et", "iso639_2": "EST", "label": "Estonian", "uri": "http://publications.europa.eu/resource/authority/language/EST"},
    {"code": "lv", "iso639_2": "LAV", "label": "Latvian", "uri": "http://publications.europa.eu/resource/authority/language/LAV"},
    {"code": "lt", "iso639_2": "LIT", "label": "Lithuanian", "uri": "http://publications.europa.eu/resource/authority/language/LIT"},
    {"code": "mt", "iso639_2": "MLT", "label": "Maltese", "uri": "http://publications.europa.eu/resource/authority/language/MLT"},
]

# Create DataFrame
df_dcat3_languages = pd.DataFrame(basic_languages)

print(f"✅ Created DCAT 3 language mapping with {len(df_dcat3_languages)} languages")
print("\nDCAT 3 Language Mapping:")
print(df_dcat3_languages.to_string(index=False))

# Save to CSV
df_dcat3_languages.to_csv("dcat3_language_mapping.csv", index=False)
print(f"\n✅ Saved to 'dcat3_language_mapping.csv'")

# Create lookup functions for DCAT 3
def lookup_language_by_iso639_1(code):
    """Look up language by ISO 639-1 code (e.g., 'en', 'fr')"""
    return df_dcat3_languages[df_dcat3_languages["code"] == code.lower()]

def lookup_language_by_iso639_2(code):
    """Look up language by ISO 639-2 code (e.g., 'ENG', 'FRA')"""
    return df_dcat3_languages[df_dcat3_languages["iso639_2"] == code.upper()]

def lookup_language_by_name(name):
    """Look up language by name"""
    return df_dcat3_languages[df_dcat3_languages["label"].str.contains(name, case=False)]

# Test the functions
print(f"\n=== TESTING DCAT 3 LANGUAGE LOOKUP FUNCTIONS ===")
print("lookup_language_by_iso639_1('en'):")
print(lookup_language_by_iso639_1('en').to_string(index=False))

print(f"\nlookup_language_by_iso639_2('FRA'):")
print(lookup_language_by_iso639_2('FRA').to_string(index=False))

print(f"\nlookup_language_by_name('German'):")
print(lookup_language_by_name('German').to_string(index=False))

=== ALTERNATIVE APPROACH: Different Language Authority Endpoints ===
Trying alternative endpoints...

Trying: https://publications.europa.eu/resource/authority/language.rdf
✅ Successfully loaded 0 triples
❌ No language concepts with prefLabels found

Trying: https://publications.europa.eu/resource/authority/language.xml
✅ Successfully loaded 0 triples
❌ No language concepts with prefLabels found

Trying: https://op.europa.eu/en/web/eu-vocabularies/authority-tables
❌ Failed to load: sequence item 0: expected str instance, NoneType found

=== MANUAL DCAT 3 LANGUAGE MAPPING ===
Since the RDF endpoints may not provide complete vocabulary,
creating a basic ISO 639-1/639-2 language mapping for DCAT 3:
✅ Created DCAT 3 language mapping with 23 languages

DCAT 3 Language Mapping:
code iso639_2      label                                                           uri
  en      ENG    English http://publications.europa.eu/resource/authority/language/ENG
  fr      FRA     French http://publication

In [ ]:
# DCAT 3 Usage Example with Correct Language Data
print("=== DCAT 3 USAGE EXAMPLE ===")

# Example: How to use the language data in DCAT 3 RDF
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, DCTERMS

# Create a sample DCAT 3 dataset with proper language information
print("Creating sample DCAT 3 dataset with proper language metadata...")

# Create graph and namespaces
dcat_graph = Graph()
DCAT = Namespace("http://www.w3.org/ns/dcat#")
FOAF = Namespace("http://xmlns.com/foaf/0.1/")

# Bind prefixes for readability
dcat_graph.bind("dcat", DCAT)
dcat_graph.bind("dct", DCTERMS)
dcat_graph.bind("foaf", FOAF)

# Create a sample dataset
dataset_uri = URIRef("https://example.org/dataset/my-multilingual-dataset")

# Add basic dataset information
dcat_graph.add((dataset_uri, RDF.type, DCAT.Dataset))
dcat_graph.add((dataset_uri, DCTERMS.title, Literal("My Multilingual Dataset", lang="en")))
dcat_graph.add((dataset_uri, DCTERMS.description, Literal("A sample dataset with multiple language versions", lang="en")))

# Add language information using the correct EU authority URIs
english_lang_uri = URIRef("http://publications.europa.eu/resource/authority/language/ENG")

# Add language properties
dcat_graph.add((dataset_uri, DCTERMS.language, english_lang_uri))

# Create distributions in different languages
en_distribution = URIRef("https://example.org/dataset/my-multilingual-dataset/distribution/en")
dcat_graph.add((dataset_uri, DCAT.distribution, en_distribution))


# English distribution
dcat_graph.add((en_distribution, RDF.type, DCAT.Distribution))
dcat_graph.add((en_distribution, DCTERMS.title, Literal("English Version", lang="en")))
dcat_graph.add((en_distribution, DCTERMS.language, english_lang_uri))
dcat_graph.add((en_distribution, DCAT.mediaType, Literal("text/csv")))


print("✅ Created sample DCAT 3 dataset with proper language metadata")

# Serialize and display the RDF
print("\n=== GENERATED DCAT 3 RDF (Turtle format) ===")
turtle_output = dcat_graph.serialize(format="turtle")
print(turtle_output)

# Save to file
with open("dcat3_sample_with_languages.ttl", "w", encoding="utf-8") as f:
    f.write(turtle_output)

print("✅ Saved sample DCAT 3 RDF to 'dcat3_sample_with_languages.ttl'")

print(f"\n=== SUMMARY FOR DCAT 3 IMPLEMENTATION ===")
print("For DCAT 3 dct:language properties, use these URIs:")
print("• English: http://publications.europa.eu/resource/authority/language/ENG")  
print("• French: http://publications.europa.eu/resource/authority/language/FRA")
print("• German: http://publications.europa.eu/resource/authority/language/DEU")
print("• Spanish: http://publications.europa.eu/resource/authority/language/SPA")
print("• etc. (see the CSV file for complete mapping)")
print(f"\n📁 Files created:")
print("• dcat3_language_mapping.csv - Complete language mapping")
print("• dcat3_sample_with_languages.ttl - Sample DCAT 3 RDF with languages")

=== DCAT 3 USAGE EXAMPLE ===
Creating sample DCAT 3 dataset with proper language metadata...
✅ Created sample DCAT 3 dataset with proper language metadata

=== GENERATED DCAT 3 RDF (Turtle format) ===
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dct: <http://purl.org/dc/terms/> .

<https://example.org/dataset/my-multilingual-dataset> a dcat:Dataset ;
    dct:description "A sample dataset with multiple language versions"@en ;
    dct:language <http://publications.europa.eu/resource/authority/language/ENG> ;
    dct:title "My Multilingual Dataset"@en ;
    dcat:distribution <https://example.org/dataset/my-multilingual-dataset/distribution/en> .

<https://example.org/dataset/my-multilingual-dataset/distribution/en> a dcat:Distribution ;
    dct:language <http://publications.europa.eu/resource/authority/language/ENG> ;
    dct:title "English Version"@en ;
    dcat:mediaType "text/csv" .


✅ Saved sample DCAT 3 RDF to 'dcat3_sample_with_languages.ttl'

=== SUMMARY FOR DCAT 3 IMPLEME

In [9]:
turtle_output

'@prefix dcat: <http://www.w3.org/ns/dcat#> .\n@prefix dct: <http://purl.org/dc/terms/> .\n\n<https://example.org/dataset/my-multilingual-dataset> a dcat:Dataset ;\n    dct:description "A sample dataset with multiple language versions"@en ;\n    dct:language <http://publications.europa.eu/resource/authority/language/ENG> ;\n    dct:title "My Multilingual Dataset"@en ;\n    dcat:distribution <https://example.org/dataset/my-multilingual-dataset/distribution/en> .\n\n<https://example.org/dataset/my-multilingual-dataset/distribution/en> a dcat:Distribution ;\n    dct:language <http://publications.europa.eu/resource/authority/language/ENG> ;\n    dct:title "English Version"@en ;\n    dcat:mediaType "text/csv" .\n\n'